In [1]:
#!/usr/bin/env python
# coding: utf-8


import os
#Directory here
os.chdir('/Users/cherryleheu/Codes/NIDIS-Codes/H-RIP/Python')


import geopandas as gpd
import shapely 
from shapely.geometry import Point
import matplotlib.pyplot as plt
# from dms2dec.dms_convert import dms2dec
import numpy as np
import pandas as pd
import rasterio
# from rasterio.plot import show
from rasterstats import zonal_stats
from datetime import datetime, timedelta
import calendar
from dateutil.relativedelta import *
from standard_precip import spi

In [5]:
# Read the shapefile
gdf = gpd.read_file('./shapefiles/RID.shp')
thisYear=(datetime.today().strftime("%Y"))
thisMonth=int((datetime.today().strftime("%m")))
# yestYr, yestMonth, yestDay = int((datetime.today() + relativedelta(days=-1)).strftime("%Y")),int((datetime.today() + relativedelta(days=-1)).strftime("%m")),int((datetime.today() + relativedelta(days=-1)).strftime("%d"))
months = np.arange(1,13,1)

for index, row in gdf.iterrows():
    ranchid = row['Polygon']
    geom = row.geometry
    temp = pd.DataFrame({'Year': [],'Month':[],'tmean_f': []})
    for i in np.arange(1990,int(thisYear)+1):
        for j in months:
            if int(i) == int(thisYear) and int(j)==(thisMonth):
                break
            with rasterio.open(f'/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean/tmean_{i}_{j:02d}.tif') as src:
                affine = src.transform
                array = src.read(1)
                df_zonal_stats = pd.DataFrame(zonal_stats(geom, array, affine=affine,nodata=-9999,stats = ['mean']))
            temp_f = 9/5*df_zonal_stats['mean'].iloc[0]+32
            new_row = pd.DataFrame({'Year':i,'Month':j,'tmean_f':temp_f},index=[0])
            temp=pd.concat([temp, new_row],ignore_index=True)
    temp['datetime']=pd.date_range('1/1/1990','1/1/2026',freq='MS')
    temp.to_csv(f'../RID/{ranchid}/{ranchid}_tmean.csv',index=False)

In [ ]:
gdf = gpd.read_file('./shapefiles/RID.shp')

datem = datetime.today().strftime("%m")
monthInd = -int(datem)+1

def temp_avg(arr):
    tdf = arr.groupby(['Month'],as_index=False)['tmean_f'].mean()
    tdf['Month'] = tdf['Month'].astype(np.uint8).apply(lambda x: calendar.month_name[x])
    tdf['rolledMonth'] = np.roll(tdf.Month, monthInd) 
    tdf['NewTemp'] = np.roll(tdf.tmean_f, monthInd)
    tdf = tdf.rename(columns={"Month":"oldMonth","Temp":"OldTemp","NewTemp": "Temp","rolledMonth":"Month"})
    return tdf

def temp_12m(arr):
    tdf12m = arr
    tdf12m['Month'] = tdf12m.tail(12)['Month'].astype(np.uint8).apply(lambda x: calendar.month_name[x])
    tdf12m = arr.tail(12)
    return tdf12m

temp = pd.read_csv(f"../RID/RID{r:03d}/RID{r:03d}_tmean.csv")    
tdf=temp_avg(temp)
tdf12m=temp_12m(temp)
tdf12m=tdf12m.drop(['Year','datetime'],axis=1)
tdf.to_csv(f'../RID/RID{r:03d}/RID{r:03d}_t_month.csv') 
tdf12m.to_csv(f'../RID/RID{r:03d}/RID{r:03d}_t_12m.csv')